# Output BAB 4.1.2 — Verifikasi Kesesuaian Grid dan Stacking Multisensor

Notebook ini khusus membuat artefak untuk subbab **4.1.2 Verifikasi Kesesuaian Grid dan Stacking Multisensor** sesuai `TODO_BAB_4_Lengkap.md` dan `TODO_BAB_4_Checklist_Output.md`. Output mencakup tabel alignment per wilayah, verifikasi band `stack_7ch.tif`, overlay OpenStreetMap, panel tile multi-layer dari lokasi yang sama, narasi interpretasi, dan checklist cakupan TODO.


## 0. Setup

Konfigurasi path, folder output, plotting style, dan helper toleran untuk membaca raster/tile.


In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import rasterio
except Exception as exc:
    rasterio = None
    RASTERIO_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
else:
    RASTERIO_IMPORT_ERROR = ""

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "font.size": 10,
})

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = ROOT / "dataset"
RUNS = ROOT / "runs"
OUT = ROOT / "outputs" / "bab4"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"
NARRATIVE_DIR = OUT / "narratives"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
NARRATIVE_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    "Aceh_Besar", "Aceh_Tamiang", "Aceh_Timur", "Aceh_Utara", "Agam",
    "Banda_Aceh", "Bireuen", "Langsa", "Pasaman_Barat", "Pidie", "Pidie_Jaya",
]
TEST_REGION = "Aceh_Utara"
CHANNEL_NAMES = ["vv_norm", "vh_norm", "hue", "saturation", "value", "slope_norm", "hand_norm"]

print(f"ROOT: {ROOT}")
print(f"Tables: {TABLE_DIR}")
print(f"Figures: {FIG_DIR}")
print("rasterio:", "available" if rasterio else f"missing ({RASTERIO_IMPORT_ERROR})")


In [ ]:
def display_df(df: pd.DataFrame, max_rows: int = 20):
    if df.empty:
        display(df)
    else:
        display(df.head(max_rows))


def save_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved table: {path.relative_to(ROOT)} ({len(df)} rows)")
    display_df(df)
    return path


def save_fig(fig, name: str) -> Path:
    path = FIG_DIR / name
    try:
        fig.tight_layout(rect=[0, 0, 1, 0.96])
    except Exception:
        fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    print(f"saved figure: {path.relative_to(ROOT)}")
    plt.show()
    return path


def read_csv_or_status(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "missing", "note": label}])
    try:
        return pd.read_csv(path)
    except Exception as exc:
        return pd.DataFrame([{"source": str(path.relative_to(ROOT)), "status": "read_error", "note": f"{type(exc).__name__}: {exc}"}])


def safe_div(num: float, den: float) -> float:
    return float(num / den) if den else 0.0


def pct(num: float, den: float) -> float:
    return round(100.0 * safe_div(num, den), 4)


def finite_stats(arr: np.ndarray, sample_step: int = 1) -> dict:
    a = np.asarray(arr)
    if sample_step > 1 and a.ndim >= 2:
        a = a[..., ::sample_step, ::sample_step]
    a = a.astype("float64", copy=False)
    finite = np.isfinite(a)
    vals = a[finite]
    if vals.size == 0:
        return {"count": 0, "valid_pct": 0.0, "min": np.nan, "max": np.nan, "mean": np.nan, "std": np.nan, "p2": np.nan, "p98": np.nan}
    return {
        "count": int(vals.size),
        "valid_pct": round(100.0 * vals.size / a.size, 4),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "p2": float(np.percentile(vals, 2)),
        "p98": float(np.percentile(vals, 98)),
    }


def normalize_image(arr: np.ndarray, p_low: float = 2, p_high: float = 98) -> np.ndarray:
    a = np.asarray(arr, dtype="float32")
    finite = np.isfinite(a)
    if not finite.any():
        return np.zeros_like(a, dtype="float32")
    lo, hi = np.percentile(a[finite], [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(a[finite])), float(np.nanmax(a[finite]))
    if hi <= lo:
        return np.zeros_like(a, dtype="float32")
    return np.clip((a - lo) / (hi - lo), 0, 1)


def raster_stats(path: Path, band: int = 1, sample_step: int = 10) -> dict:
    if rasterio is None:
        return {"status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}
    if not path.exists():
        return {"status": "missing", "note": str(path.relative_to(ROOT))}
    try:
        with rasterio.open(path) as src:
            arr = src.read(band, masked=True)
            vals = arr.compressed() if np.ma.isMaskedArray(arr) else np.asarray(arr).ravel()
            if sample_step > 1:
                arr2 = np.asarray(arr.filled(np.nan) if np.ma.isMaskedArray(arr) else arr)
                vals = arr2[::sample_step, ::sample_step].ravel()
            stats = finite_stats(vals)
            stats.update({
                "status": "ok",
                "height": src.height,
                "width": src.width,
                "crs": str(src.crs),
                "transform": str(src.transform),
            })
            return stats
    except Exception as exc:
        return {"status": "read_error", "note": f"{type(exc).__name__}: {exc}"}


def load_npz(path: Path):
    return np.load(path, allow_pickle=False)


def first_npz(path: Path) -> Path | None:
    files = sorted(path.glob("*.npz"))
    return files[0] if files else None


def parse_tile_rc(path: Path) -> tuple[int | None, int | None]:
    m = re.search(r"_r(\d+)_c(\d+)", path.stem)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def choose_tile(region: str = TEST_REGION) -> Path | None:
    tile_dir = DATASET / "tiles" / "7ch" / "by_region" / region
    candidates = []
    for path in sorted(tile_dir.glob("*.npz")):
        try:
            z = load_npz(path)
            y = np.asarray(z["y"])[0]
            valid = np.asarray(z.get("valid_mask", np.ones_like(y)))[0]
            positives = int(((y > 0) & (valid > 0)).sum())
            valid_count = int((valid > 0).sum())
            candidates.append((positives, valid_count, path))
        except Exception:
            continue
    if not candidates:
        return None
    positives = [c for c in candidates if c[0] > 0]
    if positives:
        return sorted(positives, key=lambda x: x[0], reverse=True)[0][2]
    return sorted(candidates, key=lambda x: x[1], reverse=True)[0][2]


def find_matching_prediction(model: str, tile_path: Path) -> Path | None:
    pred_dir = RUNS / "final" / model / "eval_test" / "predictions" / TEST_REGION
    candidate = pred_dir / tile_path.name
    if candidate.exists():
        return candidate
    r, c = parse_tile_rc(tile_path)
    if r is None:
        return first_npz(pred_dir)
    matches = sorted(pred_dir.glob(f"*r{r:06d}_c{c:06d}.npz"))
    return matches[0] if matches else first_npz(pred_dir)


## 4.1.2 Tabel Alignment Raster Multisensor

Mengecek ukuran raster, resolusi, CRS/proyeksi, dan geotransform setiap layer terhadap raster referensi Sentinel-1 VV.


In [ ]:
def raster_identity(path: Path) -> dict:
    if rasterio is None:
        return {"exists": path.exists(), "status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}
    if not path.exists():
        return {"exists": False, "status": "missing", "note": str(path.relative_to(ROOT))}
    try:
        with rasterio.open(path) as src:
            return {
                "exists": True,
                "status": "ok",
                "height": src.height,
                "width": src.width,
                "shape": f"{src.height}x{src.width}",
                "pixel_resolution": f"{abs(src.transform.a):.6g} x {abs(src.transform.e):.6g}",
                "crs": str(src.crs),
                "transform": tuple(round(v, 9) for v in src.transform[:6]),
            }
    except Exception as exc:
        return {"exists": path.exists(), "status": "read_error", "note": f"{type(exc).__name__}: {exc}"}


feature_layers = ["vv_norm.tif", "vh_norm.tif", "hue.tif", "saturation.tif", "value.tif", "slope_norm.tif", "hand_norm.tif"]
alignment_rows = []
raster_meta_rows = []
for region in REGIONS:
    feature_dir = DATASET / "features_preprocessed" / region
    label_dir = DATASET / "labels_unosat_rasterized" / region
    reference_path = feature_dir / "vv_norm.tif"
    ref = raster_identity(reference_path)
    checked_paths = [("Sentinel-1 VV reference", reference_path)]
    checked_paths.extend((layer, feature_dir / layer) for layer in feature_layers if layer != "vv_norm.tif")
    checked_paths.append(("stack_7ch.tif", feature_dir / "stack_7ch.tif"))
    checked_paths.append(("label_flood_binary.tif", label_dir / "label_flood_binary.tif"))

    shape_matches = []
    crs_matches = []
    transform_matches = []
    missing_layers = []
    for layer_name, layer_path in checked_paths:
        ident = raster_identity(layer_path)
        raster_meta_rows.append({
            "region": region,
            "raster": layer_name,
            "path": str(layer_path.relative_to(ROOT)),
            **{k: v for k, v in ident.items() if k != "transform"},
            "transform": str(ident.get("transform", "")),
        })
        if ident.get("status") != "ok" or ref.get("status") != "ok":
            missing_layers.append(layer_name)
            continue
        shape_matches.append(ident.get("height") == ref.get("height") and ident.get("width") == ref.get("width"))
        crs_matches.append(ident.get("crs") == ref.get("crs"))
        transform_matches.append(ident.get("transform") == ref.get("transform"))

    all_shape = bool(shape_matches) and all(shape_matches)
    all_crs = bool(crs_matches) and all(crs_matches)
    all_transform = bool(transform_matches) and all(transform_matches)
    alignment_rows.append({
        "wilayah": region,
        "raster_referensi": "Sentinel-1 VV (vv_norm.tif)",
        "ukuran_raster": ref.get("shape", ""),
        "resolusi_piksel": ref.get("pixel_resolution", ""),
        "crs_proyeksi": ref.get("crs", ""),
        "jumlah_layer_dicek": len(checked_paths),
        "shape_match": "Sama" if all_shape else "Tidak sama/perlu cek",
        "crs_match": "Sama" if all_crs else "Tidak sama/perlu cek",
        "geotransform_match": "Sama" if all_transform else "Tidak sama/perlu cek",
        "layer_bermasalah": "; ".join(missing_layers),
        "status_alignment": "Selaras" if all_shape and all_crs and all_transform and not missing_layers else "Perlu cek",
    })

alignment = pd.DataFrame(alignment_rows)
save_table(alignment, "4_1_2_alignment_verification.csv")

raster_meta = pd.DataFrame(raster_meta_rows)
save_table(raster_meta, "4_1_2_raster_metadata_optional.csv")


## 4.1.2 Verifikasi Layer pada `stack_7ch.tif`

Membuktikan band 1-7 pada `stack_7ch.tif` sejajar dengan layer individual VV, VH, HSV, Slope, dan HAND.


In [ ]:
# stack_rows = []
# layer_files = ["vv_norm.tif", "vh_norm.tif", "hue.tif", "saturation.tif", "value.tif", "slope_norm.tif", "hand_norm.tif"]
# layer_names = ["VV", "VH", "Hue", "Saturation", "Value", "Slope", "HAND"]
# for region in REGIONS:
#     fdir = DATASET / "features_preprocessed" / region
#     stack_path = fdir / "stack_7ch.tif"
#     if rasterio is None:
#         stack_rows.append({"region": region, "status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"})
#         continue
#     if not stack_path.exists():
#         stack_rows.append({"region": region, "status": "missing_stack", "path": str(stack_path.relative_to(ROOT))})
#         continue
#     try:
#         with rasterio.open(stack_path) as stack:
#             for idx, (layer, layer_name) in enumerate(zip(layer_files, layer_names), start=1):
#                 layer_path = fdir / layer
#                 if not layer_path.exists():
#                     stack_rows.append({"region": region, "band": idx, "nama_layer": layer_name, "status_alignment": "missing_layer"})
#                     continue
#                 with rasterio.open(layer_path) as src:
#                     same_shape = src.width == stack.width and src.height == stack.height
#                     same_crs = str(src.crs) == str(stack.crs)
#                     same_transform = src.transform.almost_equals(stack.transform)
#                     sample = stack.read(idx, out_shape=(min(stack.height, 512), min(stack.width, 512)), masked=True)
#                     values = sample.compressed() if np.ma.isMaskedArray(sample) else np.asarray(sample).ravel()
#                     band_stats = finite_stats(values)
#                     stack_rows.append({
#                         "region": region,
#                         "band": idx,
#                         "nama_layer": layer_name,
#                         "sumber": layer,
#                         "stack_path": str(stack_path.relative_to(ROOT)),
#                         "layer_path": str(layer_path.relative_to(ROOT)),
#                         "ukuran_raster": f"{stack.height}x{stack.width}",
#                         "resolusi_piksel": f"{abs(stack.transform.a):.6g} x {abs(stack.transform.e):.6g}",
#                         "crs_proyeksi": str(stack.crs),
#                         "shape_match": same_shape,
#                         "crs_match": same_crs,
#                         "geotransform_match": same_transform,
#                         "status_alignment": "Selaras" if same_shape and same_crs and same_transform else "Perlu cek",
#                         "mean_sample": band_stats.get("mean"),
#                         "std_sample": band_stats.get("std"),
#                     })
#     except Exception as exc:
#         stack_rows.append({"region": region, "status_alignment": "read_error", "note": f"{type(exc).__name__}: {exc}"})
# stack_verification = pd.DataFrame(stack_rows)
# save_table(stack_verification, "4_1_2_stack_7ch_layer_verification.csv")


## 4.1.2 Overlay OpenStreetMap

Membuat overlay OSM terhadap VV, VH, HSV/pseudo-RGB, Slope, HAND, dan label UNOSAT pada tile representatif Aceh_Utara.


In [ ]:
# if rasterio is None:
#     osm_status_table = pd.DataFrame([{"region": TEST_REGION, "status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}])
#     save_table(osm_status_table, "4_1_2_osm_overlay_status.csv")
# else:
#     from rasterio.windows import Window
#     from rasterio.plot import plotting_extent
#     try:
#         from pyproj import Transformer
#         import geopandas as gpd
#         import osmnx as ox
#         OSM_IMPORT_ERROR = ""
#     except Exception as exc:
#         Transformer = None
#         gpd = None
#         ox = None
#         OSM_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

#     def read_stack_window_for_tile(tile_path: Path, region: str):
#         r, c = parse_tile_rc(tile_path)
#         if r is None or c is None:
#             r = c = 0
#         stack_path = DATASET / "features_preprocessed" / region / "stack_7ch.tif"
#         label_path = DATASET / "labels_unosat_rasterized" / region / "label_flood_binary.tif"
#         with rasterio.open(stack_path) as src:
#             window = Window(c, r, min(512, src.width - c), min(512, src.height - r))
#             arr = src.read(window=window, boundless=True, fill_value=0)
#             transform = src.window_transform(window)
#             extent = plotting_extent(arr[0], transform)
#             crs = src.crs
#             bounds = rasterio.windows.bounds(window, src.transform)
#         label = None
#         if label_path.exists():
#             with rasterio.open(label_path) as lab:
#                 label = lab.read(1, window=window, boundless=True, fill_value=0)
#         return arr, label, extent, bounds, crs

#     def fetch_osm_features(bounds_projected, crs):
#         if OSM_IMPORT_ERROR:
#             return None, f"import_error: {OSM_IMPORT_ERROR}"
#         left, bottom, right, top = bounds_projected
#         transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
#         west, south = transformer.transform(left, bottom)
#         east, north = transformer.transform(right, top)
#         bbox = (west, south, east, north)
#         tags = {"highway": True, "building": True, "waterway": True, "natural": ["water"], "landuse": ["residential", "industrial", "commercial"]}
#         try:
#             gdf = ox.features_from_bbox(bbox, tags)
#             if gdf.empty:
#                 return gdf.to_crs(crs), "empty"
#             return gdf.to_crs(crs), "ok"
#         except Exception as exc:
#             empty = gpd.GeoDataFrame(geometry=[], crs=crs) if gpd is not None else None
#             return empty, f"fetch_error: {type(exc).__name__}: {exc}"

#     osm_rows = []
#     tile_path = choose_tile(TEST_REGION)
#     if tile_path is None:
#         osm_rows.append({"region": TEST_REGION, "status": "missing_tile"})
#     else:
#         try:
#             arr, label, extent, bounds, crs = read_stack_window_for_tile(tile_path, TEST_REGION)
#             osm_gdf, osm_status = fetch_osm_features(bounds, crs)
#             backgrounds = [
#                 (normalize_image(arr[0]), "(a) OSM + VV", "gray"),
#                 (normalize_image(arr[1]), "(b) OSM + VH", "gray"),
#                 (np.dstack([normalize_image(arr[4]), normalize_image(arr[3]), normalize_image(arr[2])]), "(c) OSM + HSV/pseudo-RGB", None),
#                 (normalize_image(arr[5]), "(d) OSM + Slope", "magma"),
#                 (normalize_image(arr[6]), "(e) OSM + HAND", "viridis"),
#                 (normalize_image(label) if label is not None else np.zeros(arr.shape[1:]), "(f) OSM + label UNOSAT", "Blues"),
#             ]
#             fig, axes = plt.subplots(2, 3, figsize=(13, 8))
#             axes = axes.ravel()
#             for ax, (bg, title, cmap) in zip(axes, backgrounds):
#                 if cmap:
#                     ax.imshow(bg, extent=extent, origin="upper", cmap=cmap)
#                 else:
#                     ax.imshow(bg, extent=extent, origin="upper")
#                 if osm_gdf is not None and not osm_gdf.empty:
#                     line_like = osm_gdf[osm_gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])]
#                     poly_like = osm_gdf[osm_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
#                     if not poly_like.empty:
#                         poly_like.boundary.plot(ax=ax, color="#FFD166", linewidth=0.6, alpha=0.8)
#                     if not line_like.empty:
#                         line_like.plot(ax=ax, color="#EF476F", linewidth=0.7, alpha=0.9)
#                 ax.set_title(title)
#                 ax.set_xlim(extent[0], extent[1])
#                 ax.set_ylim(extent[2], extent[3])
#                 ax.axis("off")
#             fig.suptitle(f"Overlay OSM pada Stack Preprocessing - {tile_path.stem}", y=0.99)
#             save_fig(fig, "4_1_2_osm_overlay_stack_aceh_utara.png")
#             osm_rows.append({"region": TEST_REGION, "tile": tile_path.name, "status": osm_status, "features": 0 if osm_gdf is None else len(osm_gdf), "figure": "outputs/bab4/figures/4_1_2_osm_overlay_stack_aceh_utara.png"})
#         except Exception as exc:
#             osm_rows.append({"region": TEST_REGION, "status": "render_error", "note": f"{type(exc).__name__}: {exc}"})
#     osm_status_table = pd.DataFrame(osm_rows)
#     save_table(osm_status_table, "4_1_2_osm_overlay_status.csv")


## 4.1.2 Overlay OpenStreetMap - Full Wilayah Aceh Utara

Membuat overlay OSM terhadap VV, VH, HSV/pseudo-RGB, Slope, HAND, dan label UNOSAT pada seluruh wilayah Aceh Utara (decimated/downsampled 10x untuk visualisasi).


In [ ]:
# if rasterio is None:
#     osm_status_table_full = pd.DataFrame([{"region": TEST_REGION, "status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"}])
#     save_table(osm_status_table_full, "4_1_2_osm_overlay_status_full.csv")
# else:
#     from rasterio.plot import plotting_extent
#     try:
#         from pyproj import Transformer
#         import geopandas as gpd
#         import osmnx as ox
#         OSM_IMPORT_ERROR = ""
#     except Exception as exc:
#         Transformer = None
#         gpd = None
#         ox = None
#         OSM_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

#     def read_stack_full_region(region: str, decimate: int = 10):
#         stack_path = DATASET / "features_preprocessed" / region / "stack_7ch.tif"
#         label_path = DATASET / "labels_unosat_rasterized" / region / "label_flood_binary.tif"
#         with rasterio.open(stack_path) as src:
#             new_height = src.height // decimate
#             new_width = src.width // decimate
#             arr = src.read(out_shape=(src.count, new_height, new_width), resampling=rasterio.enums.Resampling.bilinear)
#             extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
#             crs = src.crs
#             bounds = (src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)
#         label = None
#         if label_path.exists():
#             with rasterio.open(label_path) as lab:
#                 label = lab.read(1, out_shape=(new_height, new_width), resampling=rasterio.enums.Resampling.nearest)
#         return arr, label, extent, bounds, crs

#     def fetch_osm_features_full(bounds_projected, crs):
#         if OSM_IMPORT_ERROR:
#             return None, f"import_error: {OSM_IMPORT_ERROR}"
#         left, bottom, right, top = bounds_projected
#         transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
#         west, south = transformer.transform(left, bottom)
#         east, north = transformer.transform(right, top)
#         bbox = (west, south, east, north)
#         tags = {"highway": True, "waterway": True, "natural": ["water"]}
#         try:
#             gdf = ox.features_from_bbox(bbox, tags)
#             if gdf.empty:
#                 return gdf.to_crs(crs), "empty"
#             return gdf.to_crs(crs), "ok"
#         except Exception as exc:
#             empty = gpd.GeoDataFrame(geometry=[], crs=crs) if gpd is not None else None
#             return empty, f"fetch_error: {type(exc).__name__}: {exc}"

#     osm_rows = []
#     try:
#         arr, label, extent, bounds, crs = read_stack_full_region(TEST_REGION, decimate=10)
#         osm_gdf, osm_status = fetch_osm_features_full(bounds, crs)
#         backgrounds = [
#             (normalize_image(arr[0]), "(a) OSM + VV (Full)", "gray"),
#             (normalize_image(arr[1]), "(b) OSM + VH (Full)", "gray"),
#             (np.dstack([normalize_image(arr[4]), normalize_image(arr[3]), normalize_image(arr[2])]), "(c) OSM + HSV/pseudo-RGB (Full)", None),
#             (normalize_image(arr[5]), "(d) OSM + Slope (Full)", "magma"),
#             (normalize_image(arr[6]), "(e) OSM + HAND (Full)", "viridis"),
#             (normalize_image(label) if label is not None else np.zeros(arr.shape[1:]), "(f) OSM + label UNOSAT (Full)", "Blues"),
#         ]
#         fig, axes = plt.subplots(2, 3, figsize=(13, 8))
#         axes = axes.ravel()
#         for ax, (bg, title, cmap) in zip(axes, backgrounds):
#             if cmap:
#                 ax.imshow(bg, extent=extent, origin="upper", cmap=cmap)
#             else:
#                 ax.imshow(bg, extent=extent, origin="upper")
#             if osm_gdf is not None and not osm_gdf.empty:
#                 line_like = osm_gdf[osm_gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])]
#                 poly_like = osm_gdf[osm_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
#                 if not poly_like.empty:
#                     poly_like.boundary.plot(ax=ax, color="#FFD166", linewidth=0.5, alpha=0.7)
#                 if not line_like.empty:
#                     line_like.plot(ax=ax, color="#EF476F", linewidth=0.5, alpha=0.8)
#             ax.set_title(title)
#             ax.set_xlim(extent[0], extent[1])
#             ax.set_ylim(extent[2], extent[3])
#             ax.axis("off")
#         fig.suptitle(f"Overlay OSM pada Preprocessed Stack - Full Wilayah {TEST_REGION}", y=0.99)
#         save_fig(fig, "4_1_2_osm_overlay_stack_aceh_utara_full.png")
#         osm_rows.append({"region": TEST_REGION, "tile": "full_region", "status": osm_status, "features": 0 if osm_gdf is None else len(osm_gdf), "figure": "outputs/bab4/figures/4_1_2_osm_overlay_stack_aceh_utara_full.png"})
#     except Exception as exc:
#         osm_rows.append({"region": TEST_REGION, "status": "render_error", "note": f"{type(exc).__name__}: {exc}"})
#     osm_status_table_full = pd.DataFrame(osm_rows)
#     save_table(osm_status_table_full, "4_1_2_osm_overlay_status_full.csv")


## 4.1.2 Panel Tile Multi-layer

Menampilkan satu tile yang sama pada seluruh channel utama untuk memeriksa konsistensi lokasi antar-layer.


In [ ]:
tile_path = choose_tile(TEST_REGION)
panel_rows = []
if rasterio is None:
    panel_rows.append({"region": TEST_REGION, "status": "skipped", "note": f"rasterio unavailable: {RASTERIO_IMPORT_ERROR}"})
elif tile_path is None:
    panel_rows.append({"region": TEST_REGION, "status": "missing_tile"})
else:
    try:
        arr, label, extent, bounds, crs = read_stack_window_for_tile(tile_path, TEST_REGION)
        panels = [
            (normalize_image(arr[0]), "(a) VV", "gray"),
            (normalize_image(arr[1]), "(b) VH", "gray"),
            (np.dstack([normalize_image(arr[4]), normalize_image(arr[3]), normalize_image(arr[2])]), "(c) HSV/pseudo-RGB", None),
            (normalize_image(arr[5]), "(d) Slope", "magma"),
            (normalize_image(arr[6]), "(e) HAND", "viridis"),
            (normalize_image(label) if label is not None else np.zeros(arr.shape[1:]), "(f) Label UNOSAT", "Blues"),
        ]
        fig, axes = plt.subplots(2, 3, figsize=(12, 7.4))
        axes = axes.ravel()
        for ax, (image, title, cmap) in zip(axes, panels):
            if cmap:
                ax.imshow(image, cmap=cmap, vmin=0, vmax=1)
            else:
                ax.imshow(image)
            ax.set_title(title)
            ax.set_xticks([])
            ax.set_yticks([])
        fig.suptitle(f"Panel Tile Multi-layer pada Lokasi Sama - {tile_path.stem}")
        save_fig(fig, "4_1_2_same_tile_multilayer_panel_aceh_utara.png")
        panel_rows.append({
            "region": TEST_REGION,
            "tile": tile_path.name,
            "status": "ok",
            "figure": "outputs/bab4/figures/4_1_2_same_tile_multilayer_panel_aceh_utara.png",
        })
    except Exception as exc:
        panel_rows.append({"region": TEST_REGION, "tile": tile_path.name, "status": "render_error", "note": f"{type(exc).__name__}: {exc}"})

same_tile_panel_status = pd.DataFrame(panel_rows)
save_table(same_tile_panel_status, "4_1_2_same_tile_panel_status.csv")


## 4.1.2 Narasi Interpretasi

Menyimpan paragraf siap pakai untuk menjelaskan bukti alignment, fungsi OSM sebagai pendukung visual, dan konsekuensi jika layer bergeser.


In [ ]:
aligned_count = 0
if "alignment" in globals() and not alignment.empty and "status_alignment" in alignment.columns:
    aligned_count = int((alignment["status_alignment"] == "Selaras").sum())
total_regions = len(REGIONS)
stack_ok = False
if "stack_verification" in globals() and not stack_verification.empty and "status_alignment" in stack_verification.columns:
    stack_ok = bool((stack_verification["status_alignment"] == "Selaras").all())

interpretation_text = f"""# Interpretasi Verifikasi Alignment 4.1.2

Verifikasi grid dilakukan dengan menjadikan raster Sentinel-1 VV sebagai referensi. Setiap layer fitur, `stack_7ch.tif`, dan label UNOSAT dicek terhadap ukuran raster, CRS/proyeksi, serta geotransform referensi. Dari {total_regions} wilayah, {aligned_count} wilayah berstatus selaras pada tabel alignment.

Bukti utama alignment adalah kesamaan dimensi, CRS, dan geotransform. Overlay OpenStreetMap dipakai sebagai bukti visual pendukung untuk memastikan pola sungai, jaringan jalan, pesisir, dan permukiman berada pada posisi yang konsisten terhadap VV, VH, HSV, Slope, HAND, dan label UNOSAT.

Apabila salah satu layer bergeser, model akan menerima pasangan fitur-label dari lokasi yang berbeda. Kondisi tersebut dapat membuat model mempelajari hubungan spasial yang salah, misalnya mengaitkan label banjir dengan sinyal radar atau topografi dari piksel tetangga yang tidak sesuai.

Hasil verifikasi `stack_7ch.tif` menunjukkan status band stack: {'selaras untuk seluruh band dan wilayah' if stack_ok else 'masih perlu dicek pada beberapa baris tabel'}. Dengan demikian, stack multisensor layak digunakan sebagai input model selama baris status alignment tetap `Selaras`.
"""

narrative_path = NARRATIVE_DIR / "4_1_2_alignment_interpretation.md"
narrative_path.write_text(interpretation_text, encoding="utf-8")
print(f"saved narrative: {narrative_path.relative_to(ROOT)}")
print(interpretation_text)
narrative_path


## Checklist 4.1.2

Memverifikasi artefak 4.1.2 yang diminta TODO.


In [ ]:
expected = [
    ("wajib", "Tabel verifikasi alignment raster per wilayah", TABLE_DIR / "4_1_2_alignment_verification.csv"),
    ("disarankan", "Tabel teknis verifikasi layer dalam stack_7ch.tif", TABLE_DIR / "4_1_2_stack_7ch_layer_verification.csv"),
    ("wajib", "Gambar overlay OSM terhadap VV, VH, HSV, Slope, HAND, dan label UNOSAT", FIG_DIR / "4_1_2_osm_overlay_stack_aceh_utara.png"),
    ("sangat_disarankan", "Panel satu tile multi-layer dari lokasi yang sama", FIG_DIR / "4_1_2_same_tile_multilayer_panel_aceh_utara.png"),
    ("narasi_wajib", "Penjelasan bukti utama alignment dan peran OSM sebagai pendukung", NARRATIVE_DIR / "4_1_2_alignment_interpretation.md"),
    ("wajib", "Status render overlay OSM", TABLE_DIR / "4_1_2_osm_overlay_status.csv"),
    ("wajib", "Status render panel tile multi-layer", TABLE_DIR / "4_1_2_same_tile_panel_status.csv"),
]
checklist_4_1_2 = pd.DataFrame([
    {
        "priority": priority,
        "todo_item": todo_item,
        "artifact": str(path.relative_to(ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for priority, todo_item, path in expected
])
save_table(checklist_4_1_2, "4_1_2_todo_coverage_checklist.csv")
missing = checklist_4_1_2[~checklist_4_1_2["exists"]]
if missing.empty:
    print("All 4.1.2 TODO artifacts created.")
else:
    print("Missing 4.1.2 artifacts:")
    display(missing)
